# Ireland Work Permits — Exploratory Notebook

**Project:** Ireland Employment Map
**Data source:** Department of Enterprise, Trade & Employment (DETE) + Irish Immigration Service Delivery (ISD)
**Coverage:** Work permits 2015–2025 · Long-term visa decisions 2017–2025
**Stack:** pandas · SQLite · Plotly

---

This notebook walks through the full analysis pipeline in one place:

1. Load the cleaned CSVs produced by `src/01_clean_data.py`
2. Query them with SQL via an in-memory SQLite database
3. Visualise the results with interactive Plotly charts
4. Build the final choropleth map (see `output/map/` or the [live demo](http://ireland-employment-map-adarsh-kodali.s3-website-eu-west-1.amazonaws.com))

> **Prerequisites** — run once before opening this notebook:
> ```bash
> python src/01_clean_data.py   # cleans raw Excel files → data/cleaned/
> ```
> Everything else (SQLite, charts, map) is done inline below.

## Setup

In [ ]:
import os, sqlite3
from pathlib import Path
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Move to project root if running from the notebooks/ subdirectory.
# All relative paths (data/cleaned/, output/, src/) are anchored to the root.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(f'Working directory: {Path.cwd()}')

# ── Paths ──────────────────────────────────────────────────────────────────
CLEANED_DIR = Path('data/cleaned')

# ── Colour palette (Ireland flag colours) ─────────────────────────────────
IRELAND_GREEN  = '#169B62'
IRELAND_ORANGE = '#FF883E'
TEMPLATE       = 'plotly_white'

# ── Helper: run SQL and return a DataFrame ─────────────────────────────────
# Wraps pd.read_sql_query() to keep analytical cells concise.
def q(sql, conn):
    return pd.read_sql_query(sql, conn)

---
## 1. The Cleaned Data

`src/01_clean_data.py` normalised 40+ messy Excel files into four tidy CSVs.  
Let's peek at each one before we start querying.

In [ ]:
# county_permits.csv — one row per (year, county)
county_df = pd.read_csv(CLEANED_DIR / 'county_permits.csv')
print(f'Shape    : {county_df.shape[0]:,} rows x {county_df.shape[1]} cols')
print(f'Years    : {sorted(county_df["year"].unique())}')
print(f'Counties : {sorted(county_df["county"].unique())}')
county_df.head(6)

In [ ]:
# sector_permits.csv — one row per (year, sector)
sector_df = pd.read_csv(CLEANED_DIR / 'sector_permits.csv')
print(f'Shape   : {sector_df.shape[0]:,} rows x {sector_df.shape[1]} cols')
print(f'Years   : {sorted(sector_df["year"].unique())}')
print(f'Sectors : {sector_df["sector"].nunique()} unique (after name normalisation)')
sector_df.head(4)

In [ ]:
# nationality_permits.csv — one row per (year, nationality)
nat_df = pd.read_csv(CLEANED_DIR / 'nationality_permits.csv')
print(f'Shape         : {nat_df.shape[0]:,} rows x {nat_df.shape[1]} cols')
print(f'Years         : {sorted(nat_df["year"].unique())}')
print(f'Nationalities : {nat_df["nationality"].nunique()} unique')
nat_df.head(4)

In [ ]:
# visa_decisions.csv — long-term visa applications only (student / employment / graduate)
# Short-term tourist visas were excluded in 01_clean_data.py via an allow list.
visa_df = pd.read_csv(CLEANED_DIR / 'visa_decisions.csv')
print(f'Shape         : {visa_df.shape[0]:,} rows x {visa_df.shape[1]} cols')
print(f'Years         : {sorted(visa_df["year"].unique())}')
print(f'Nationalities : {visa_df["nationality"].nunique()} unique')
visa_df.head(4)

---
## 2. Load into SQLite

We load all four CSVs into an **in-memory SQLite database**.
This lets us write expressive SQL instead of chaining pandas operations,
and mirrors `src/02_build_sqlite.py` (which builds the on-disk `.db` file).

> **Why in-memory?** SQLite needs to create and delete journal files when writing.
> An in-memory database (`:memory:`) avoids any file-system restrictions,
> lives entirely in RAM, and supports all the same SQL as a real database file.

In [ ]:
conn = sqlite3.connect(':memory:')

tables = [
    ('county_permits.csv',      'county_permits'),
    ('sector_permits.csv',      'sector_permits'),
    ('nationality_permits.csv', 'nationality_permits'),
    ('visa_decisions.csv',      'visa_decisions'),
]

for fname, tname in tables:
    df_tmp = pd.read_csv(CLEANED_DIR / fname)
    # to_sql writes the full DataFrame into SQLite as a table.
    # index=False omits pandas' row-number index from the table.
    df_tmp.to_sql(tname, conn, if_exists='replace', index=False)
    print(f'  loaded {tname:<25} {len(df_tmp):>6,} rows')

print('\nAll tables loaded. Ready to query.')

---
## 3. National Trend (2015–2025)

How has the total number of work permits issued across Ireland changed year-on-year?

In [ ]:
# SUM(issued) adds up all 26 county values for each year = national total.
national_df = q("""
    SELECT
        year,
        SUM(issued)    AS total_issued,
        SUM(refused)   AS total_refused,
        SUM(withdrawn) AS total_withdrawn
    FROM   county_permits
    GROUP  BY year
    ORDER  BY year
""", conn)

# .diff()       = difference between each row and the row above (NaN for first row)
# .pct_change() = same but as a fraction; multiply by 100 for %
national_df['yoy_change'] = national_df['total_issued'].diff().astype('Int64')
national_df['yoy_pct']    = (national_df['total_issued'].pct_change() * 100).round(1)

national_df

In [ ]:
# Dual-axis chart:
#   Left axis  → total permits issued (filled area line)
#   Right axis → year-on-year % change (bar)
# Combining both shows volume AND growth momentum at a glance —
# the YoY bars make the 2022 doubling jump out while the area line
# shows the longer-running level shift.

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(
    x=national_df['year'], y=national_df['total_issued'],
    name='Permits issued',
    mode='lines+markers',
    line=dict(color=IRELAND_GREEN, width=3),
    marker=dict(size=8),
    fill='tozeroy',
    fillcolor='rgba(22,155,98,0.12)',
), secondary_y=False)

# Green bars for positive YoY growth, orange for negative.
# The 2020 bar is tiny (≈0%) — COVID didn't really show up in this dataset.
fig.add_trace(go.Bar(
    x=national_df['year'], y=national_df['yoy_pct'],
    name='YoY % change',
    marker_color=[
        IRELAND_GREEN if (v and v >= 0) else IRELAND_ORANGE
        for v in national_df['yoy_pct'].fillna(0)
    ],
    opacity=0.55,
), secondary_y=True)

fig.update_layout(
    title='Ireland Work Permits Issued — National Total (2015–2025)',
    xaxis=dict(title='Year', tickmode='linear', dtick=1),
    yaxis=dict(title='Total permits issued'),
    yaxis2=dict(title='Year-on-year % change', ticksuffix='%'),
    template=TEMPLATE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified',
)
fig.show()

**Key findings — National Trend**

- Permits grew steadily from 2015 through 2019, then **plateaued — not collapsed — through the pandemic**. 2019, 2020 and 2021 were within ~1% of each other, so COVID barely registered in the headline numbers.
- The real shock was **2022**, when national totals more than doubled in a single year (≈16k → ≈40k). Whatever was paused during the pandemic seems to have unfrozen all at once.
- **2024** is the most recent complete year. **2025** shows a broad cooldown — back to roughly 2023 levels — suggesting 2024 was a peak rather than the new normal.

> **Limitation:** The data counts permits *issued*, not unique active workers.
> A worker who renewed their permit is counted again each year.

---
## 4. County Rankings & National Share

Which counties attract the most work permit holders, and how has that changed?

In [ ]:
# SQL JOIN: calculate each county's share of the national total.
# The inner sub-query ('t') computes the national total per year.
# JOIN ON c.year = t.year attaches that total to every county row —
# like a VLOOKUP in Excel — so we can compute pct_share per row.
county_share_df = q("""
    SELECT
        c.year,
        c.county,
        c.issued,
        c.refused,
        c.withdrawn,
        ROUND(100.0 * c.issued / t.total, 2) AS pct_share
    FROM county_permits c
    JOIN (
        SELECT year, SUM(issued) AS total
        FROM   county_permits
        GROUP  BY year
    ) t ON c.year = t.year
    ORDER BY c.year, c.issued DESC
""", conn)

top5_2024 = county_share_df[county_share_df['year'] == 2024].head(5)
print('Top 5 counties in 2024:')
print(top5_2024[['county', 'issued', 'pct_share']].to_string(index=False))

In [ ]:
# Horizontal bar chart: top 10 counties in 2024.
# Text label on each bar = county's % share of national total.
top10 = (
    county_share_df[county_share_df['year'] == 2024]
    .nlargest(10, 'issued')
    .sort_values('issued')   # ascending so largest bar appears at the top
)

fig = px.bar(
    top10, x='issued', y='county', orientation='h',
    text=top10['pct_share'].apply(lambda v: f'{v:.1f}%'),
    color='issued',
    color_continuous_scale=[[0, '#d4f0e3'], [1, IRELAND_GREEN]],
    labels={'issued': 'Permits issued', 'county': 'County'},
    title='Top 10 Counties by Work Permits Issued (2024)',
    template=TEMPLATE,
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, xaxis_title='Permits issued', yaxis_title='')
fig.show()

In [ ]:
# Line chart: how the top 6 counties (ranked by 2024 totals) changed over 2015–2025.
top6_counties = (
    county_share_df[county_share_df['year'] == 2024]
    .nlargest(6, 'issued')['county'].tolist()
)
trend_df = county_share_df[county_share_df['county'].isin(top6_counties)]

fig = px.line(
    trend_df, x='year', y='issued', color='county',
    markers=True,
    labels={'issued': 'Permits issued', 'year': 'Year', 'county': 'County'},
    title='Work Permit Trends — Top 6 Counties (2015–2025)',
    template=TEMPLATE,
)
fig.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),
    hovermode='x unified',
    legend_title='County',
)
fig.show()

---
## 5. County Growth (2015 → 2024)

Which counties grew the fastest? Which shrank?

In [ ]:
# SQL self-join: match each county's 2015 value against its 2024 value.
# Query 'a' = 2015 data. Query 'b' = 2024 data.
# INNER JOIN ON county keeps only counties present in BOTH years.
growth_df = q("""
    SELECT
        a.county,
        a.issued                                            AS issued_2015,
        b.issued                                            AS issued_2024,
        (b.issued - a.issued)                               AS abs_change,
        ROUND(100.0*(b.issued - a.issued) / a.issued, 1)   AS pct_change
    FROM
        (SELECT county, issued FROM county_permits WHERE year = 2015) a
    JOIN
        (SELECT county, issued FROM county_permits WHERE year = 2024) b
        ON a.county = b.county
    ORDER BY pct_change DESC
""", conn)

print(f'Counties with 2015→2024 comparison: {len(growth_df)}')
growth_df

In [ ]:
# Diverging horizontal bar chart: green = growth, orange = decline.
# Dashed vertical line at 0 is the dividing line.
df_sorted = growth_df.sort_values('pct_change')

fig = px.bar(
    df_sorted, x='pct_change', y='county', orientation='h',
    color='pct_change',
    color_continuous_scale=[
        [0.0, IRELAND_ORANGE],
        [0.5, '#ffffcc'],
        [1.0, IRELAND_GREEN],
    ],
    labels={'pct_change': '% change', 'county': 'County'},
    title='Work Permit Growth by County (2015 → 2024)',
    template=TEMPLATE,
    hover_data={'issued_2015': True, 'issued_2024': True},
)
fig.update_layout(
    coloraxis_showscale=False,
    xaxis_title='% change in permits issued',
    yaxis_title='',
)
fig.add_vline(x=0, line_dash='dash', line_color='#aaa')
fig.show()

---
## 6. Sector Breakdown (2020–2025)

Which economic sectors issue the most work permits?

> **Why only from 2020?** DETE renamed all sector categories around 2020
> (e.g. `Agriculture & Fisheries` → `A - Agriculture, Forestry & Fishing`).
> Mixing old and new names produces misleading trend lines, so sector analysis
> is restricted to 2020–2025 where names are consistent.

> **Why national only?** DETE does not publish a county-level sector breakdown,
> so the map's sector filter (and the chart below) are national totals.

In [ ]:
sector_analysis_df = q("""
    SELECT year, sector, issued
    FROM   sector_permits
    WHERE  year >= 2020
    ORDER  BY year, issued DESC
""", conn)

print(f'Rows: {len(sector_analysis_df):,}')
print(f'Unique sectors: {sector_analysis_df["sector"].nunique()}')
sector_analysis_df[sector_analysis_df['year'] == 2024].head(6)[['sector', 'issued']]

In [ ]:
# Top 10 sectors in 2024 — strip the NACE letter prefix (e.g. 'J - ')
# so axis labels are shorter and more readable.
top10_sectors = (
    sector_analysis_df[sector_analysis_df['year'] == 2024]
    .nlargest(10, 'issued')
    .sort_values('issued')
    .copy()
)
top10_sectors['sector_short'] = top10_sectors['sector'].str.replace(
    r'^[A-Z]\s*-\s*', '', regex=True
).str.strip()

fig = px.bar(
    top10_sectors, x='issued', y='sector_short', orientation='h',
    color='issued',
    color_continuous_scale=[[0, '#d4f0e3'], [1, IRELAND_GREEN]],
    labels={'issued': 'Permits issued', 'sector_short': 'Sector'},
    title='Top 10 Sectors by Work Permits Issued (2024)',
    template=TEMPLATE,
)
fig.update_layout(coloraxis_showscale=False, xaxis_title='Permits issued', yaxis_title='')
fig.show()

In [ ]:
# Trend line: how the top 6 sectors changed from 2020 to 2025.
top6_sectors = (
    sector_analysis_df[sector_analysis_df['year'] == 2024]
    .nlargest(6, 'issued')['sector'].tolist()
)
sector_trend_df = sector_analysis_df[sector_analysis_df['sector'].isin(top6_sectors)].copy()
sector_trend_df['sector_short'] = sector_trend_df['sector'].str.replace(
    r'^[A-Z]\s*-\s*', '', regex=True
).str.strip()

fig = px.line(
    sector_trend_df, x='year', y='issued', color='sector_short',
    markers=True,
    labels={'issued': 'Permits issued', 'year': 'Year', 'sector_short': 'Sector'},
    title='Work Permit Trends — Top 6 Sectors (2020–2025)',
    template=TEMPLATE,
)
fig.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),
    hovermode='x unified',
    legend_title='Sector',
)
fig.show()

---
## 7. Source Nationalities (2024)

Where do work permit holders come from?

In [ ]:
nat_analysis_df = q("""
    SELECT nationality, issued, refused, withdrawn
    FROM   nationality_permits
    WHERE  year = 2024
    ORDER  BY issued DESC
    LIMIT  15
""", conn)

nat_analysis_df

In [ ]:
fig = px.bar(
    nat_analysis_df.sort_values('issued'),
    x='issued', y='nationality', orientation='h',
    color='issued',
    color_continuous_scale=[[0, '#d4f0e3'], [1, IRELAND_GREEN]],
    labels={'issued': 'Permits issued', 'nationality': 'Nationality'},
    title='Top 15 Source Nationalities — Work Permits Issued (2024)',
    template=TEMPLATE,
)
fig.update_layout(coloraxis_showscale=False, xaxis_title='Permits issued', yaxis_title='')
fig.show()

---
## 8. Visa Decisions — Long-Term Only (2017–2025)

This dataset comes from the Irish Immigration Service Delivery (ISD).  
We only look at **long-term visa applications** (student, employment, graduate visas).  
Short-term tourist/visitor visas were excluded in `src/01_clean_data.py` via an allow list.

In [ ]:
# Annual totals: received, granted, refused — and grant rate %.
# NULLIF(received, 0) prevents divide-by-zero; NULL / anything = NULL in SQL.
visa_trend_df = q("""
    SELECT
        year,
        SUM(received) AS total_received,
        SUM(granted)  AS total_granted,
        SUM(refused)  AS total_refused
    FROM   visa_decisions
    WHERE  year < 2026
    GROUP  BY year
    ORDER  BY year
""", conn)

# Compute grant rate in Python to safely handle NaN
total_rcv = visa_trend_df['total_received']
has_data  = total_rcv.notna() & (total_rcv > 0)
visa_trend_df['grant_rate_pct'] = (
    (visa_trend_df['total_granted'] / total_rcv * 100).where(has_data).round(1)
)

print(visa_trend_df[['year','total_received','total_granted','grant_rate_pct']].to_string(index=False))

In [ ]:
# Dual-axis chart:
#   Left  → stacked bars (granted=green, refused=orange)
#   Right → grant rate % dotted line

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=visa_trend_df['year'], y=visa_trend_df['total_granted'],
    name='Granted', marker_color=IRELAND_GREEN, opacity=0.85,
), secondary_y=False)

fig.add_trace(go.Bar(
    x=visa_trend_df['year'], y=visa_trend_df['total_refused'],
    name='Refused', marker_color=IRELAND_ORANGE, opacity=0.85,
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=visa_trend_df['year'], y=visa_trend_df['grant_rate_pct'],
    name='Grant rate %',
    mode='lines+markers',
    line=dict(color='#333333', width=2, dash='dot'),
    marker=dict(size=7),
), secondary_y=True)

fig.update_layout(
    barmode='stack',
    title='Ireland Long-Term Visa Decisions — Annual Trend (2017–2025)',
    xaxis=dict(title='Year', tickmode='linear', dtick=1),
    yaxis=dict(title='Visa applications'),
    yaxis2=dict(title='Grant rate %', ticksuffix='%', range=[0, 100]),
    template=TEMPLATE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified',
)
fig.show()

In [ ]:
# Top 20 nationalities by applications, coloured by grant rate %.
# Green = high approval, orange = high refusal.
visa_rates_df = q("""
    SELECT
        nationality,
        received,
        granted,
        refused,
        ROUND(100.0 * granted / NULLIF(received, 0), 1) AS grant_rate_pct
    FROM   visa_decisions
    WHERE  year = 2024
      AND  received IS NOT NULL
    ORDER  BY received DESC
    LIMIT  20
""", conn)

visa_rates_df.head(8)

In [ ]:
fig = px.bar(
    visa_rates_df.sort_values('received'),
    x='received', y='nationality', orientation='h',
    color='grant_rate_pct',
    color_continuous_scale=[
        [0.0, IRELAND_ORANGE],
        [0.5, '#ffffcc'],
        [1.0, IRELAND_GREEN],
    ],
    range_color=[0, 100],
    labels={
        'received': 'Applications received',
        'nationality': 'Nationality',
        'grant_rate_pct': 'Grant rate %',
    },
    title='Long-Term Visa Applications & Approval Rates by Nationality (2024)',
    template=TEMPLATE,
    hover_data={'granted': True, 'refused': True, 'grant_rate_pct': True},
)
fig.update_layout(
    xaxis_title='Applications received', yaxis_title='',
    coloraxis_colorbar=dict(title='Grant rate %', ticksuffix='%'),
)
fig.show()

In [ ]:
# Top 15 nationalities by long-term visas GRANTED in 2024.
# Text label on each bar = grant rate %.
visa_top_df = q("""
    SELECT nationality, received, granted, refused,
           ROUND(100.0 * granted / NULLIF(received, 0), 1) AS grant_rate_pct
    FROM   visa_decisions
    WHERE  year = 2024
      AND  granted IS NOT NULL
    ORDER  BY granted DESC
    LIMIT  15
""", conn)

df_sorted = visa_top_df.sort_values('granted')

fig = px.bar(
    df_sorted, x='granted', y='nationality', orientation='h',
    color='granted',
    color_continuous_scale=[[0, '#d4f0e3'], [1, IRELAND_GREEN]],
    text=df_sorted['grant_rate_pct'].apply(lambda v: f'{v}%' if pd.notna(v) else ''),
    labels={'granted': 'Visas granted', 'nationality': 'Nationality'},
    title='Top 15 Nationalities — Long-Term Visas Granted (2024)',
    template=TEMPLATE,
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, xaxis_title='Visas granted', yaxis_title='')
fig.show()

---
## 9. Build the Interactive Choropleth Map

The final output is a fully self-contained HTML map — no server required.
Run the cell below to regenerate `output/map/ireland_employment_map.html`.

The map includes a two-handle year-range slider (default 2019–2025), a sector
filter, and a sidebar that updates with top counties and the sector breakdown
for whatever range/sector you select. Sector data is national-level only —
the map itself always shows total county permits.

In [ ]:
# %run executes src/04_build_map.py and streams its output here.
# It is equivalent to: python src/04_build_map.py (from the project root).
%run src/04_build_map.py

In [ ]:
# After the cell above completes, open the map in your browser.
import subprocess
map_path = Path('output/map/ireland_employment_map.html').resolve()
print(f'Map saved to: {map_path}')
# Uncomment to auto-open (macOS):
# subprocess.run(['open', str(map_path)])

---
## Key Findings

| Theme | Finding |
|-------|---------|
| **COVID barely registered** | 2019, 2020 and 2021 national totals were within ~1% of each other. The real shock was 2022, when permits more than doubled (≈16k → ≈40k) in a single year. |
| **Healthcare leads, with one exception** | Health & Social Work has been the top sector every year since 2020 — except **2022**, when a tech-hiring surge briefly pushed Information & Communication ahead (10.8k vs 9.8k). By 2024 the healthcare lead had widened to roughly 2× IT. |
| **Agriculture rebound** | Agriculture, Forestry & Fishing more than doubled from 2023 to 2024 after an unexplained dip the year before — pointing to growing demand for rural hires. |
| **Quiet county winners** | Meath went from 72 permits in 2015 to over 1,500 in 2024. Monaghan from 20 to 500+. Neither is a tech hub, which makes the growth more interesting. |
| **2025 cooldown** | National permits fell from ≈39k in 2024 to ≈31k in 2025 — back to 2023 levels. Only five counties grew (Kilkenny +34%, Laois +19%, Donegal +14%, Wicklow +5%, Leitrim +48% off a small base); Kildare (-49%) and Waterford (-46%) fell hardest. |
| **Top source nationalities** | India and Brazil have been the top two source nationalities in recent years, by a wide margin. |
| **Visa approvals** | Long-term visa grant rates ranged ~60–80% from 2017 to 2025, with notable variation by nationality. |

### Assumptions & Limitations

- **Permit counts ≠ unique workers.** Renewals are counted each year alongside new permits.
- **Sector data only from 2020.** DETE renamed sectors around 2020; pre-2020 names can't be reliably matched.
- **Sector data is national-level only.** DETE does not publish a county-level sector breakdown.
- **2025 data is partial** — the year was still in progress when this dataset was downloaded; treat 2025 figures as a snapshot, not a full-year total.
- **Suppressed visa counts** (marked `*` in source) are treated as `NaN`, not zero.
- **County data = Republic of Ireland only.** Northern Ireland counties excluded via allowlist in `01_clean_data.py`.

---
*Data sources: DETE · ISD · simplemaps.com (GeoJSON)*
*Live demo: [ireland-employment-map-adarsh-kodali.s3-website-eu-west-1.amazonaws.com](http://ireland-employment-map-adarsh-kodali.s3-website-eu-west-1.amazonaws.com)*

In [ ]:
# Close the in-memory SQLite connection.
# Good practice even though it disappears when the kernel stops.
conn.close()
print('SQLite connection closed. Done.')